In [1]:
import json
from pathlib import Path

import stable_whisper
from langchain_text_splitters import RecursiveCharacterTextSplitter
import ollama
from tqdm import tqdm
from llama_cpp import Llama
import yt_dlp

import stable_whisper
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.rankers import MetaFieldRanker
from haystack.components.samplers import TopPSampler
from haystack import Pipeline

/home/jobin/Projects/transcript_summarizer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def download_audio(url: str, save_dir: Path, save_name: str) -> str:
    output_path = Path(f"{save_dir}/{save_name}.mp3")
    with yt_dlp.YoutubeDL({'extract_audio': True, 'format': 'bestaudio', 'outtmpl': f'{output_path}'}) as video:
        video.download(url)
    return str(output_path)
    
def postprocess_chunk(chunk: str) -> str:
    chunk = chunk.lstrip(". ")
    chunk = chunk.capitalize()
    chunk += "."
    return chunk

def ollama_chat(model_name: str, prompt: str, format: str = "") -> str:
    response = ollama.chat(model=model_name,
                           messages=[
                                        {
                                            'role': 'user',
                                            'content': prompt,
                                        },
                                    ],
                           stream=False,
                           format=format)
    return response["message"]["content"]

def load_llm(gguf_path: str) -> Llama:
    llm = Llama(model_path=gguf_path, chat_format="chatml", verbose=False)
    return llm

def llamacpp_chat(llm, system_prompt: str, user_prompt: str, schema: dict) -> str:
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_object",
            "schema": schema,
        },
        temperature=0.0,
    )
    return response["choices"][0]["message"]["content"]

def summarize_chunk(chunk: str) -> str:
    return ollama_chat("llama3_instruct_podcast_chunk_summarizer", chunk)

BULLETIZATION_SYSTEM_PROMPT = """You are a helpful assistant that outputs in JSON. You are an expert at converting text into short points. Convert the given text into short points and a title. Don't add escape characters. Directly list the points and the title, don't add additional text before or after it."""
BULLETIZATION_SCHEMA = {
                            "type": "object",
                            "properties": {"title": {"type": "string"},
                                            "bullets": {"type": "array", "items": {"type": "string"}}
                                            },
                            "required": ["title", "bullets"],
                        }

def bulletize_summary(summary: str, llm: Llama) -> str:
    response_text = llamacpp_chat(llm, 
                                  system_prompt=BULLETIZATION_SYSTEM_PROMPT, 
                                  user_prompt=summary,
                                  schema=BULLETIZATION_SCHEMA)
    return json.loads(response_text)

In [3]:
TEST_FILE_PATH = download_audio(url="https://www.youtube.com/watch?v=0m3hGZvD-0s",
                                save_dir="/home/jobin/Projects/multimodal_rag_summarization/data",
                                save_name="test2")

[youtube] Extracting URL: https://www.youtube.com/watch?v=0m3hGZvD-0s
[youtube] 0m3hGZvD-0s: Downloading webpage


[youtube] 0m3hGZvD-0s: Downloading ios player API JSON
[youtube] 0m3hGZvD-0s: Downloading android player API JSON


[youtube] 0m3hGZvD-0s: Downloading player edea0cc6
[youtube] 0m3hGZvD-0s: Downloading m3u8 information
[info] 0m3hGZvD-0s: Downloading 1 format(s): 251
[download] Destination: /home/jobin/Projects/multimodal_rag_summarization/data/test2.mp3
[download] 100% of   26.76MiB in 00:00:03 at 6.69MiB/s   


In [4]:
tiny_whisper = stable_whisper.load_model("tiny")
result = tiny_whisper.transcribe(TEST_FILE_PATH, word_timestamps=True)
result_dict = result.to_dict()
transcript = result_dict["text"].strip()

Transcribe:   0%|          | 0/2142.76 [00:00<?, ?sec/s]

Detected language: english


Transcribe: 100%|█████████▉| 2142.75/2142.76 [00:37<00:00, 57.84sec/s]


In [5]:
splitter = RecursiveCharacterTextSplitter(separators=["."],
                                          chunk_size=5000,
                                          chunk_overlap=0,
                                          length_function=len,
                                          is_separator_regex=False)

chunks = splitter.split_text(transcript)
chunks = [postprocess_chunk(x) for x in chunks]

In [7]:
llm = load_llm(gguf_path="/home/jobin/Projects/transcript_summarizer/gguf/Meta-Llama-3-8B-Instruct.Q6_K.gguf")
summary_json_list = []
for chunk in tqdm(chunks):
    summary = summarize_chunk(chunk)
    summary_json = bulletize_summary(summary, llm)
    summary_json_list.append(summary_json)

100%|██████████| 7/7 [02:20<00:00, 20.12s/it]


In [8]:
summary_json_list

[{'title': 'Daily Routine for Maximizing Productivity',
  'bullets': ['Wake up at 6:30 am',
   'Morning mantra includes setting rules for social media use, diet, and exercise',
   "Mantra also involves gratitude, visualizing goals, and focusing on the day's tasks"]},
 {'title': 'Daily Routine',
  'bullets': ['Morning mantra sets the tone for the day',
   'Four-hour focused session of deep work with minimal interruptions',
   'Standing desk and ergonomic keyboard setup',
   'Love for music, particularly guitar playing',
   'Long run at the end of the day with brown noise and audiobook',
   'Reflecting on thoughts and emotions during the run']},
 {'title': 'Daily Routine and Reflections',
  'bullets': ['Running and bodyweight exercises while fasting helps me focus mentally and physically.',
   'I enjoy exercising fasted as it allows me to tap into my mental and physical toughness.',
   'Reflecting on history, particularly Nazi Germany, makes me realize the importance of mental toughness 

In [15]:
def merge_segments_by_idx(segments: list[dict], start_idx: int, end_idx: int) -> dict:
    if start_idx == end_idx:
        return segments[start_idx]
    merged_segment = {}
    merged_text = ""
    merged_words = []
    no_speech_probs = []
    for idx in range(start_idx, end_idx + 1):
        merged_text += segments[idx]["text"]
        merged_words.append(segments[idx]["words"])
        no_speech_probs.append(segments[idx]["no_speech_prob"])
    merged_segment["text"] = merged_text
    merged_segment["start"] = segments[start_idx]["start"]
    merged_segment["end"] = segments[end_idx]["end"]
    merged_segment["words"] = merged_words
    merged_segment["no_speech_prob"] = no_speech_probs
    return merged_segment

def merge_segments(segments: list) -> list:
    start_idx = 0
    end_idx = 0
    merged_segments = []
    while end_idx < len(segments):
        segment = segments[end_idx]
        if segment["text"].endswith(".") or end_idx == len(segments)-1:
            merged_segment = merge_segments_by_idx(segments, start_idx, end_idx)
            merged_segments.append(merged_segment)
            start_idx = end_idx + 1
            end_idx = start_idx
        else:
            end_idx += 1
    return merged_segments

def prepare_docs(segments: list) -> list[dict]:
    docs = []
    for segment in segments:
        doc = {}
        doc["content"] = segment["text"].strip()
        
        meta_dict = {}
        meta_dict["start"] = segment["start"]
        meta_dict["end"] = segment["end"]
        meta_dict["words"] = segment["words"]
        meta_dict["no_speech_prob"] = segment["no_speech_prob"]
        doc["meta"] = meta_dict

        docs.append(doc)
    docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in docs]
    return docs

def load_embedder(embedder_type: str, **kwargs):
    if embedder_type == "SentenceTransformersDocumentEmbedder":
        embedder = SentenceTransformersDocumentEmbedder(**kwargs)
        embedder.warm_up()
    if embedder_type == "SentenceTransformersTextEmbedder":
        embedder = SentenceTransformersTextEmbedder(**kwargs)
    return embedder

def load_document_store(store_type: str):
    if store_type == "InMemoryDocumentStore":
        document_store = InMemoryDocumentStore()
    return document_store

def load_retriever(retriever_type: str, document_store, **kwargs):
    if retriever_type == "InMemoryEmbeddingRetriever":
        retriever = InMemoryEmbeddingRetriever(document_store, **kwargs)
    return retriever

def load_ranker(ranker_type: str, **kwargs):
    if ranker_type == "MetaFieldRanker":
        ranker = MetaFieldRanker(**kwargs)
    return ranker

def load_sampler(sampler_type: str, **kwargs):
    if sampler_type == "TopPSampler":
        sampler = TopPSampler(**kwargs)
    return sampler

def get_doc_embeddings(doc_embedder: SentenceTransformersDocumentEmbedder, docs: list[Document]) -> dict[str, list[Document]]:
    docs_with_embeddings = doc_embedder.run(docs)
    return docs_with_embeddings

def write_doc_embeddings_to_inmemory_store(docs_with_embeddings: list[Document], store: InMemoryDocumentStore):
    store.write_documents(docs_with_embeddings["documents"])

def preprocess_segments(segments: list[dict]) -> dict:
    merged_segments = merge_segments(segments)
    return merged_segments

def deduplicate_docs(docs: list[Document]) -> list[Document]:
    unique_ids = set()
    unique_docs = []
    for doc in docs:
        if doc.id not in unique_ids:
            unique_docs.append(doc)
            unique_ids.add(doc.id)
    return unique_docs

def load_pipeline(query_embedder, retriever, sampler) -> Pipeline:
    basic_pipeline = Pipeline()
    basic_pipeline.add_component("query_embedder", query_embedder)
    basic_pipeline.add_component("retriever", retriever)
    basic_pipeline.add_component("sampler", sampler)
    basic_pipeline.connect("query_embedder.embedding", "retriever.query_embedding")
    basic_pipeline.connect("retriever.documents", "sampler.documents")
    return basic_pipeline

def retrieve_for_query(query: str, ranker, pipeline: Pipeline) -> list[Document]:
    unique_ids = set()
    sampled_docs = []
    response = pipeline.run({"query_embedder": {"text": query}})
    for response_doc in response["sampler"]["documents"]:
        if response_doc.id not in unique_ids:
            unique_ids.add(response_doc.id)
            sampled_docs.append(response_doc)
    ranked_docs = ranker.run(documents=sampled_docs)
    return ranked_docs["documents"]

def enrich_summary_dict(summary_dict: dict, ranker, pipeline: Pipeline) -> dict:
    enriched_dict = {}
    enriched_dict["title"] = summary_dict["title"]
    enriched_dict["bullets"] = []
    summary_bullets = summary_dict["bullets"]
    for bullet in summary_bullets:
        bullet_dict = {}
        bullet_dict["bullet"] = bullet
        bullet_dict["retrieved"] = retrieve_for_query(bullet, ranker, pipeline)
        enriched_dict["bullets"].append(bullet_dict)
    return enriched_dict

In [10]:
merged_segments = preprocess_segments(result_dict["segments"])

In [12]:
docs = prepare_docs(merged_segments)
doc_embedder = load_embedder(embedder_type="SentenceTransformersDocumentEmbedder", model="sentence-transformers/all-mpnet-base-v2")
query_embedder = load_embedder(embedder_type="SentenceTransformersTextEmbedder", model="sentence-transformers/all-mpnet-base-v2")
docs_with_embeddings = get_doc_embeddings(doc_embedder, docs)
document_store = load_document_store(store_type="InMemoryDocumentStore")
document_store.write_documents(docs_with_embeddings["documents"])
retriever = load_retriever(retriever_type="InMemoryEmbeddingRetriever", document_store=document_store, top_k=3)
ranker = load_ranker(ranker_type="MetaFieldRanker", meta_field="start", sort_order="ascending")
sampler = load_sampler(sampler_type="TopPSampler", top_p=0.95)

Batches: 100%|██████████| 4/4 [00:00<00:00,  6.50it/s]


In [13]:
basic_pipeline = load_pipeline(query_embedder, retriever, sampler)

In [17]:
enriched_summary_dicts = []
for summary_dict in summary_json_list:
    enriched_dict = enrich_summary_dict(summary_dict, ranker, basic_pipeline)
    enriched_summary_dicts.append(enriched_dict)
        

Batches: 100%|██████████| 1/1 [00:00<00:00, 155.66it/s]
